# 11 - Dynamic risk and intelligent risk-based maintenance

## Learning objectives

- Distinguish static from dynamic risk estimates
- Map sensor evidence to a time-dependent probability proxy
- Calculate a dynamic FTA trend
- Compare now/wait triggers using expected costs


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Static prior -> dynamic risk

Unit 3 produced one $P(OVERFLOW)$ value. Evidence changes during operation, so a conditional risk estimate can change too. Digital-twin-based decision support updates model states and evidence regularly. The simple proxy below uses current residual and maintenance context; it is not a full sequential Bayesian filter for $P(H|E_{0:t})$.


In [ ]:
df=pd.read_csv(ROOT/'data'/'pundit_case'/'sensor_data.csv'); df['residual']=df.lt101_measured_pct-df.true_level_pct


## 2. Residual-to-probability mapping: an explicit assumption

Use a sigmoid for demonstration: a large negative residual maps to a larger sensor-fault probability proxy.

**This is not a universal physical law or a calibrated posterior.** A real application must calibrate, validate and quantify uncertainty in the mapping.


In [ ]:
def logistic(x): return 1/(1+np.exp(-x))
def residual_to_p(r,center=-2.5,scale=.8,p_min=.002,p_max=.6):
 score=logistic(-(r-center)/scale); return p_min+(p_max-p_min)*score
df['p_sensor']=residual_to_p(df.residual)
plt.plot(df.time_min,df.p_sensor); plt.xlabel('time [min]'); plt.ylabel('P(sensor fault | evidence)'); plt.grid(alpha=.3); plt.show()


## 3. Maintenance-overdue context

In this teaching model, an overdue condition can multiply the odds. This is an explicit assumption, not an automatically established causal effect.


In [ ]:
overdue=np.asarray(df.maintenance_overdue,dtype=bool)
odds=df.p_sensor/(1-df.p_sensor); odds_adj=odds*np.where(overdue,2.0,1.0); df['p_sensor_context']=odds_adj/(1+odds_adj)
plt.plot(df.time_min,df.p_sensor,label='evidence only'); plt.plot(df.time_min,df.p_sensor_context,label='+ overdue context'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 4. Dynamic FTA

Replace the static `LT101_LOW` probability with the time-dependent diagnostic proxy. Other inputs remain fixed to isolate its effect. The resulting trend is model-based, not a measured accident-frequency estimate.


In [ ]:
p_ch=.08; p_lshh=.01; p_xv=.005; p_op=.04; p_auto=1-(1-p_lshh)*(1-p_xv)
def p_overflow(p_sensor): return p_ch*p_auto*(1-(1-p_sensor)*(1-p_op))
df['p_overflow']=p_overflow(df.p_sensor_context)
plt.semilogy(df.time_min,df.p_overflow); plt.xlabel('time [min]'); plt.ylabel('dynamic P(overflow)'); plt.grid(alpha=.3); plt.show()


## 5. Probability times consequence

With a HUF 50 million consequence proxy, the expected-loss series represents one monetary decision dimension. Safety constraints can require intervention independently of this calculation.


In [ ]:
C=50_000_000; df['expected_consequence']=df.p_overflow*C
plt.plot(df.time_min,df.expected_consequence); plt.ylabel('expected consequence proxy [HUF]'); plt.xlabel('time [min]'); plt.grid(alpha=.3); plt.show()


## 6. Maintenance trigger

Compare immediate maintenance with the next opportunity. The economic trigger is where the expected waiting cost becomes larger. A defensible operational decision additionally needs comparable exposure horizons and calibrated intervention effects.


In [ ]:
Cm=250_000; prod_now=800_000; prod_window=100_000; p_after=.001
cost_now=Cm+prod_now+p_after*C
cost_wait=Cm+prod_window+df.p_overflow*C
df['maintain_now_cheaper']=cost_now<cost_wait
first=df.loc[df.maintain_now_cheaper,'time_min'].min(); print('first economic trigger [min]=',first,'cost now=',cost_now)


In [ ]:
plt.plot(df.time_min,cost_wait,label='wait'); plt.axhline(cost_now,ls='--',label='maintain now'); plt.xlabel('time [min]'); plt.ylabel('expected cost [HUF]'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 7. Decision chain and traceability

`sensor data -> residual -> diagnostic model -> fault probability -> FTA/Markov risk -> consequence model -> maintenance decision`.

At every interface, record the model version, calibration data, timestamp, uncertainty, assumptions and responsible component.


In [ ]:
pd.DataFrame([
 {'step':'sensor→residual','assumption':'reference level meaningful'},
 {'step':'residual→P(fault)','assumption':'calibrated evidence mapping'},
 {'step':'P(fault)→FTA','assumption':'static gate structure valid'},
 {'step':'risk→decision','assumption':'cost proxy and constraints valid'}])


## 8. Claims not justified automatically

An ARIMA or machine-learning threshold-crossing forecast is not automatically an FMEA occurrence rating or a Markov transition rate. A separate calibration model is needed. The same applies to a residual score mapped to failure probability.


## Try it!

1. Change sigmoid center/scale and inspect the risk trend.
2. Remove the overdue odds multiplier.
3. Double the consequence and find the economic trigger, if one exists.
4. Design an experiment to calibrate residual-to-fault-probability mapping.


## Summary

A dynamic risk pipeline can turn diagnostic evidence into decision-support inputs, but every conversion carries assumptions. The next unit combines the full PUNDIT teaching workflow.
